In [ ]:
import subprocess
import math
import os
import numpy as np
import polars as pl
from google.cloud import bigquery, storage
from phetk.phecode import Phecode

bigquery_client = bigquery.Client()
storage_client = storage.Client()
bucket_name = os.getenv("WORKSPACE_BUCKET").replace("gs://", "").strip("/")
workspace_cdr = os.environ["WORKSPACE_CDR"]
path_base = f"gs://{bucket_name}/disagg/data"

print(f"Target Bucket: {bucket_name}")
print(f"Workspace CDR: {workspace_cdr}")

def export_to_gcs(query, destination_path):
    uri = f"gs://{bucket_name}/{destination_path}"
    export_config = bigquery.QueryJobConfig(
        destination_encryption_configuration=None,
        use_legacy_sql=False,
    )
    export_query = (
        f"EXPORT DATA OPTIONS(uri='{uri}', format='PARQUET', overwrite=true) AS {query}"
    )
    job = bigquery_client.query(export_query, job_config=export_config)
    job.result()
    print(f"Exported: {uri}")

def gcs_file_exists(path):
    return subprocess.call(['gsutil', '-q', 'stat', path]) == 0


# In[2]:


import subprocess

subprocess.run(
    ["gsutil", "-m", "rm", "-r", f"gs://{bucket_name}/disagg/data/**"],
    check=True
)

print("Deleted disagg/data directory.")


if not gcs_file_exists(f"gs://{bucket_name}/disagg/phecode/phecode_counts_X.tsv"): 
    print("Generating Phecodes...")
    phecode_gen = Phecode(platform="aou")
    phecode_gen.count_phecode(
        phecode_version="X",
        icd_version="US",
        output_file_path=f"gs://{bucket_name}/disagg/phecode/phecode_counts_X.tsv",
    )
    print("Phecodes generated.")
else:
    print("Phecode file already exists, skip make.")


# In[3]:


person_query = f"""
SELECT
    person_id,
    age_at_consent AS age,
    sex_at_birth AS sex,
    has_ehr_data
FROM `{workspace_cdr}.cb_search_person`
"""
export_to_gcs(person_query, "disagg/data/person_*.parquet")

asian_query = f"""
SELECT
    person_id,
    survey_datetime,
    question_concept_id,
    question,
    answer
FROM `{workspace_cdr}.ds_survey`
WHERE question_concept_id IN (1586151)
"""
export_to_gcs(asian_query, "disagg/data/asian_*.parquet")

white_query = f"""
SELECT
    person_id,
    survey_datetime,
    question_concept_id,
    answer_concept_id,
    question,
    answer
FROM `{workspace_cdr}.ds_survey`
WHERE question_concept_id IN (1586140)
AND answer_concept_id IN (1586146)
"""
export_to_gcs(white_query, "disagg/data/white_*.parquet")

financial_toxicity_codes = [
    43530583,  # Couldn't afford copay
    43530585,  # Deductible too high
    43530584,  # Pay out of pocket
    43530411,  # Couldn't afford prescription meds
    43530410,  # Couldn't afford mental health care
    43528663,  # Couldn't afford emergency care
    43528662,  # Couldn't afford dental care
    43530408,  # Couldn't afford eyeglasses
    43528664,  # Couldn't afford regular doctor
    43530412,  # Couldn't afford specialist
    43530409,  # Couldn't afford follow-up care
    43530557,  # Worried about paying medical bills
    43530416,  # Skipped medication doses to save money
    43530417,  # Took less medicine to save money
    43530415,  # Delayed filling prescription
    43530413,  # Asked for lower cost medication
    43528665,  # Bought drugs from another country
    43528666,  # Used alternative therapies to save money
]
ft_query = f"""
SELECT
    person_id,
    survey_datetime,
    question_concept_id,
    question,
    answer
FROM `{workspace_cdr}.ds_survey`
WHERE question_concept_id IN ({','.join(map(str, financial_toxicity_codes))})
"""
export_to_gcs(ft_query, "disagg/data/ft_*.parquet")

covariate_codes = [
    1585375, # Income
    1585940, # Education
    1585952, # Employment
    1585386, # Insurance
    1586135, # Nativity
    1585892, # Marital Status
]
covariates_query = f"""
SELECT
    person_id,
    survey_datetime,
    question_concept_id,
    question,
    answer
FROM `{workspace_cdr}.ds_survey`
WHERE question_concept_id IN ({','.join(map(str, covariate_codes))})
"""
export_to_gcs(covariates_query, "disagg/data/covariates_*.parquet")


# In[4]:


lf_ft = pl.scan_parquet(f"{path_base}/ft_*.parquet")

QUESTION_MAP = {
    "Can't Afford Care: Alternative Therapies": "Alternative Therapies to Save Money",
    "Can't Afford Care: Emergency Care": "Emergency Care",
    "Can't Afford Care: Lower Cost Rx To Save Money": "Lower Cost Prescription",
    "Can't Afford Care: Specialist": "Specialist Care",
    "Can't Afford Care: Follow-up Care": "Follow-up Care",
    "Can't Afford Care: Mental Health Counseling": "Mental Health Care",
    "Can't Afford Care: Took Less Med To Save Money": "Took Less Medication",
    "Can't Afford Care: Dental Care": "Dental Care",
    "Can't Afford Care: Skipped Med To Save Money": "Skipped Medication",
    "Can't Afford Care: Delayed Filling Rx To Save Money": "Delayed Filling Prescription",
    "Can't Afford Care: Prescription Medicines": "Prescription Medicines",
    "Can't Afford Care: Bought Rx From Other Country": "Bought Rx From Other Country",
    "Can't Afford Care: Healthcare Provider": "Regular Doctor",
    "Can't Afford Care: Eyeglasses": "Eyeglasses",
    "Can't Afford Care: Worried About Paying": "Worried About Paying",
    "Delayed Medical Care: Deductible Too High": "Delayed Care – Deductible Too High",
    "Delayed Medical Care: Can't Afford Co-pay": "Delayed Care – Copay",
    "Delayed Medical Care: Had To Pay Out Of Pocket": "Delayed Care – Out of Pocket",
}

FT_SCORING_MAP = {
    "Alternative Therapies To Save Money: Yes": 1,
    "Bought Rx From Other Country: Yes": 1,
    "Delayed Filling Rx To Save Money: Yes": 1,
    "Can't Afford Dental Care: Yes": 1,
    "Can't Afford Emergency Care: Yes": 1,
    "Can't Afford Eyeglasses: Yes": 1,
    "Can't Afford Follow-up Care: Yes": 1,
    "Can't Afford Healthcare Provider: Yes": 1,
    "Lower Cost Rx To Save Money: Yes": 1,
    "Can't Afford Mental Health Counseling: Yes": 1,
    "Can't Afford Prescription Medicines: Yes": 1,
    "Skipped Med To Save Money: Yes": 1,
    "Can't Afford Specialist: Yes": 1,
    "Took Less Med To Save Money: Yes": 1,
    "Delayed Care Due to Can't Afford Co-pay: Yes": 1,
    "Delayed Care Due to Deductible Too High: Yes": 1,
    "Delayed Care Due to Had To Pay Out Of Pocket: Yes": 1,

    "Worried About Paying: Somewhat Worried": 1,
    "Worried About Paying: Very Worried": 1,
}

lf_ft_cleaned = (
    lf_ft
    .with_columns(
        pl.col("question").replace(QUESTION_MAP).alias("question_clean"),
        pl.col("answer")
          .replace(FT_SCORING_MAP, default=0) 
          .cast(pl.Int8)
          .alias("ft_value")
    )
)

lf_ft_reduced = (
    lf_ft_cleaned
    .group_by(["person_id", "question_clean"])
    .agg(pl.max("ft_value").alias("ft_value"))
)

question_levels = (
    lf_ft_reduced.select("question_clean").unique().sort("question_clean")
    .collect().get_column("question_clean").to_list()
)

lf_ft_wide = (
    lf_ft_reduced.pivot(
        on="question_clean",
        on_columns=question_levels,
        index="person_id",
        values="ft_value",
        aggregate_function="max",
    )
    .fill_null(0)
)

cols_crn = [
    "Skipped Medication", "Took Less Medication", 
    "Delayed Filling Prescription", "Bought Rx From Other Country"
]

cols_foregone = [
    "Delayed Care – Copay", "Delayed Care – Deductible Too High", 
    "Delayed Care – Out of Pocket", "Prescription Medicines", 
    "Specialist Care", "Follow-up Care", 
    "Alternative Therapies to Save Money" 
]

cols_subjective = ["Worried About Paying"]

lf_ft_wide = lf_ft_wide.with_columns([
    (pl.max_horizontal(cols_crn) > 0).cast(pl.Int8).alias("outcome_crn"),
    (pl.max_horizontal(cols_foregone) > 0).cast(pl.Int8).alias("outcome_foregone"),
    (pl.max_horizontal(cols_subjective) > 0).cast(pl.Int8).alias("outcome_subjective")
])


# In[5]:


lf_covariates = pl.scan_parquet(f"{path_base}/covariates_*.parquet")

COVARIATE_QUESTION_MAP = {
    "Insurance: Health Insurance": "insurance_status",
    "Income: Annual Income": "income_bracket",
    "Employment: Employment Status": "employment_status",
    "Marital Status: Current Marital Status": "marital_status",
    "Education Level: Highest Grade": "education_level",
    "The Basics: Birthplace": "nativity"
}

COVARIATE_ANSWER_MAP = {
    # Income
    "Annual Income: less 10k": "Low", "Annual Income: 10k 25k": "Low",
    "Annual Income: 25k 35k": "Low", "Annual Income: 35k 50k": "Low",
    "Annual Income: 50k 75k": "Not Low", "Annual Income: 75k 100k": "Not Low",
    "Annual Income: 100k 150k": "Not Low", "Annual Income: 150k 200k": "Not Low",
    "Annual Income: more 200k": "Not Low",
    # Education
    "Highest Grade: Never Attended": "≤HS", "Highest Grade: One Through Four": "≤HS",
    "Highest Grade: Five Through Eight": "≤HS", "Highest Grade: Nine Through Eleven": "≤HS",
    "Highest Grade: Twelve Or GED": "≤HS", "Highest Grade: College One to Three": ">HS",
    "Highest Grade: College Graduate": ">HS", "Highest Grade: Advanced Degree": ">HS",
    # Insurance
    "Health Insurance: Yes": "Insured", "Health Insurance: No": "Uninsured",
    # Employment
    "Employment Status: Employed For Wages": "Working", "Employment Status: Self Employed": "Working",
    "Employment Status: Retired": "Not working", "Employment Status: Unable To Work": "Not working",
    "Employment Status: Out Of Work Less Than One": "Not working", "Employment Status: Out Of Work One Or More": "Not working",
    "Employment Status: Student": "Not working", "Employment Status: Homemaker": "Not working",
    # Marital
    "Current Marital Status: Married": "Partnered", "Current Marital Status: Living With Partner": "Partnered",
    "Current Marital Status: Divorced": "Not partnered", "Current Marital Status: Widowed": "Not partnered",
    "Current Marital Status: Separated": "Not partnered", "Current Marital Status: Never Married": "Not partnered",
    # Nativity
    "Birthplace: USA": "US-born", "PMI: Other": "Foreign-born",
    "PMI: Skip": "Unknown",
    "PMI: Prefer Not To Answer": "Unknown",
    "PMI: Dont Know": "Unknown",
    "PMI: Don't Know": "Unknown",
}

priority_map = {"Working": 1, "Not working": 2, "Unknown": 3}

lf_covariates_wide = (
    lf_covariates
    .with_columns(
        pl.col("question").replace(COVARIATE_QUESTION_MAP).alias("question_clean"),
        pl.col("answer").replace(COVARIATE_ANSWER_MAP).alias("answer_clean")
    )
    .filter(pl.col("question_clean").is_in(COVARIATE_QUESTION_MAP.values()))
    .with_columns(
        pl.when(pl.col("question_clean") == "employment_status")
          .then(pl.col("answer_clean").replace_strict(priority_map, default=10).cast(pl.Int32))
          .otherwise(0)
          .alias("priority")
    )
    .sort(["person_id", "question_clean", "priority"])
    .unique(subset=["person_id", "question_clean"], keep="first")
    .pivot(
        on="question_clean",
        on_columns=list(COVARIATE_QUESTION_MAP.values()),
        index="person_id",
        values="answer_clean",
        aggregate_function="first"
    )
    .fill_null("Unknown")
)


lf_asian = pl.scan_parquet(f"{path_base}/asian_*.parquet")
lf_white = pl.scan_parquet(f"{path_base}/white_*.parquet")

RACE_MAPPING = {
    'Asian Specific: Cambodian': 'Cambodian', 
    'Asian Specific: Vietnamese': 'Vietnamese',
    'Asian Specific: Hmong': 'Hmong', 
    'Asian Specific: Asian Specific Indian': 'IndoPak',
    'Asian Specific: Pakistani': 'IndoPak', 
    'Asian Specific: Korean': 'Korean',
    'Asian Specific: Japanese': 'Japanese', 
    'Asian Specific: Filipino': 'Filipino',
    'Asian Specific: Chinese': 'Chinese', 
    'Asian Specific: None Of These Describe Me': 'None',
    'PMI: Skip': 'Skip'
}

all_mapped_cols = ["Cambodian", "Vietnamese", "Hmong", "IndoPak", "Korean", "Japanese", "Filipino", "Chinese", "None", "Skip"]

valid_asian_groups = ["Vietnamese", "IndoPak", "Korean", "Japanese", "Filipino", "Chinese"]

asian_df = (
    lf_asian
    .with_columns(pl.col("answer").replace(RACE_MAPPING).alias("race_clean"))
    .filter(pl.col("race_clean").is_in(all_mapped_cols))
    .collect()
    .pivot(
        on="race_clean",
        index="person_id",
        values="race_clean",
        aggregate_function="len"
    )
    .fill_null(0)
)

for col in all_mapped_cols:
    if col not in asian_df.columns:
        asian_df = asian_df.with_columns(pl.lit(0, dtype=pl.Int32).alias(col))

existing_cols = [c for c in all_mapped_cols if c in asian_df.columns]
asian_df = asian_df.with_columns([
    pl.col(c).clip(0, 1).cast(pl.Int32) for c in existing_cols
])

lf_asian_prepared = (
    asian_df.lazy()
    .select(["person_id"] + all_mapped_cols)
    .with_columns(pl.lit(0, dtype=pl.Int32).alias("white_raw"))
)

lf_white_prepared = (
    lf_white.lazy()
    .select("person_id")
    .unique()
    .with_columns(pl.lit(1, dtype=pl.Int32).alias("white_raw"))
    .with_columns([pl.lit(0, dtype=pl.Int32).alias(c) for c in all_mapped_cols])
)

race_cohort_df = (
    pl.concat([lf_asian_prepared, lf_white_prepared], how="vertical")
    .group_by("person_id")
    .agg(
        *[pl.max(col) for col in all_mapped_cols],
        pl.max("white_raw").alias("white_temp")
    )
    .with_columns(
        pl.max_horizontal(valid_asian_groups).alias("is_valid_asian")
    )
    .with_columns(
        pl.when(pl.col("is_valid_asian") == 1)
        .then(0)
        .otherwise(pl.col("white_temp"))
        .alias("White")
    )
    .filter((pl.col("White") == 1) | (pl.col("is_valid_asian") == 1))
    
    .drop(["white_temp", "is_valid_asian", "Hmong", "Cambodian", "None", "Skip"])
)


# In[6]:


COMORBIDITY_MAP = {
    "hypertension": "CV_401", "heart_failure": "CV_424", "ischemic_heart_disease": "CV_404",
    "atrial_fibrillation": "CV_416.2", "diabetes": "EM_202", "copd": "RE_474",
    "mdd": "MB_286.2", "asthma": "RE_475", "anxiety": "MB_288"
}

lf_ft_dates = (
    lf_ft.lazy()
    .group_by("person_id")
    .agg(pl.max("survey_datetime").alias("ft_index_date"))
    .with_columns(pl.col("ft_index_date").cast(pl.Date))
)

lf_phecodes_pre_ft = (
    pl.scan_csv(f"gs://{bucket_name}/disagg/phecode/phecode_counts_X.tsv", separator="\t")
    .select(["person_id", "phecode", "first_event_date"])
    .with_columns(pl.col("first_event_date").str.to_date(strict=False).alias("dx_date"))
    .join(lf_ft_dates, on="person_id", how="inner")
    .with_columns((pl.col("ft_index_date") - pl.col("dx_date")).dt.total_days().alias("days_diff"))
    .filter(pl.col("days_diff") >= 0)
)

lf_cancer_events = (
    lf_phecodes_pre_ft
    .filter(pl.col("phecode").str.starts_with("CA_") & ~pl.col("phecode").str.starts_with("CA_103.2"))
    .with_columns(pl.col("phecode").str.strip_prefix("CA_").cast(pl.Float64, strict=False).alias("p_num"))
    .filter(pl.col("p_num") < 135.0)
    .with_columns(((pl.col("p_num") >= 120.0) & (pl.col("p_num") < 126.0)).cast(pl.Int8).alias("ehr_is_hematologic"))
    
    .sort(["person_id", "dx_date"], descending=[False, True])
    .unique(subset=["person_id"], keep="first") 
)

comorb_counts = (
    lf_phecodes_pre_ft
    .group_by("person_id")
    .agg([pl.col("phecode").str.starts_with(v).any().cast(pl.Int8).alias(k) for k, v in COMORBIDITY_MAP.items()])
    .with_columns(pl.sum_horizontal(COMORBIDITY_MAP.keys()).alias("comorbidity_count"))
    .with_columns(
        pl.when(pl.col("comorbidity_count") >= 3).then(pl.lit("3+"))
        .otherwise(pl.col("comorbidity_count").cast(pl.Utf8))
        .alias("comorbidity_score_capped")
    )
)

cancer_classification = (
    lf_ft_dates
    .join(lf_cancer_events, on="person_id", how="inner")
    .with_columns([
        (pl.col("days_diff") / 365.25).alias("years_since_dx"),
        pl.col("ehr_is_hematologic").alias("is_hematologic"),
        pl.lit("ehr").alias("cancer_data_source")
    ])
    .select(["person_id", "cancer_data_source", "is_hematologic", "years_since_dx", "ft_index_date"])
)

lf_person = pl.scan_parquet(f"{path_base}/person_*.parquet")
sex_map = {"Female": "female", "Male": "male", "Unknown": "unknown"}

final_df = (
    lf_person
    .with_columns(pl.col("sex").replace(sex_map, default="unknown").alias("sex_normalized"))
    .join(lf_ft_wide.lazy(), on="person_id", how="inner")
    .join(cancer_classification, on="person_id", how="inner")
    .join(race_cohort_df.lazy(), on="person_id", how="inner")
    .join(comorb_counts.lazy(), on="person_id", how="left")
    .join(lf_covariates_wide.lazy(), on="person_id", how="left")
    .with_columns([
        pl.col("comorbidity_score_capped").fill_null("0"),
        pl.col("insurance_status").fill_null("Unknown"),
        pl.col("income_bracket").fill_null("Unknown"),
        pl.col("education_level").fill_null("Unknown")
    ])
    .collect()
)


# ==========================================================
# MISSINGNESS CHECK & MODE IMPUTATION LOGIC
# ==========================================================
print("\n" + "="*60)
print("MISSINGNESS PATTERNS & MODE IMPUTATION")
print("="*60)
total_rows = len(final_df)
check_cols = ["sex_normalized", "nativity", "marital_status", "education_level", "employment_status", "insurance_status", "income_bracket"]

# 1. Print Percentages
print("--- Percentage Unknown Before Imputation ---")
for col in check_cols:
    unk_val = "unknown" if col == "sex_normalized" else "Unknown"
    unk_count = final_df.filter(pl.col(col) == unk_val).height
    pct = (unk_count / total_rows) * 100
    print(f"{col}: {unk_count} / {total_rows} ({pct:.2f}%)")

# 2. Impute (except income)
print("\n--- Imputing Mode (Excluding Income) ---")
impute_cols = [c for c in check_cols if c != "income_bracket"]

for col in impute_cols:
    unk_val = "unknown" if col == "sex_normalized" else "Unknown"
    # Find mode ignoring unknowns
    valid_df = final_df.filter(pl.col(col) != unk_val)
    if valid_df.height > 0:
        # get the mode (if multiple, take first)
        mode_val = valid_df.select(pl.col(col).mode()).to_series()[0]
        print(f"{col} -> Mode: '{mode_val}'")
        final_df = final_df.with_columns(
            pl.when(pl.col(col) == unk_val)
            .then(pl.lit(mode_val))
            .otherwise(pl.col(col))
            .alias(col)
        )
    else:
        print(f"{col} -> Could not find mode.")
print("="*60 + "\n")
# ==========================================================


print(f"Final Study Population: {len(final_df):,}")
final_df.write_csv("final_df_disagg.csv")
print("Saved to final_df_disagg.csv")



# In[10]:


import pandas as pd
from tableone import TableOne
import warnings
import polars as pl

warnings.filterwarnings("ignore")
df_master = final_df.to_pandas()

ASIAN_COLS = [
    "Vietnamese",
    "Korean",
    "Japanese",
    "Filipino",
    "IndoPak",
    "Chinese",
]

def derive_ethnicity(row):
    if row.get("White", 0) == 1:
        return "Non-Hispanic White"
    
    asian_selected = [col for col in ASIAN_COLS if row.get(col, 0) == 1]
    
    if len(asian_selected) == 1:
        if asian_selected[0] == "IndoPak":
            return "Indo-Pakistani"
        return asian_selected[0]
    
    if len(asian_selected) > 1:
        return "Multiethnic Asian"
    
    return "Unknown/Other"

df_master["Derived_Ethnicity"] = df_master.apply(derive_ethnicity, axis=1)

rename_map = {
    "sex_normalized": "Sex",
    "nativity": "Nativity",
    "marital_status": "Marital Status",
    "education_level": "Education",
    "income_bracket": "Household Income",
    "employment_status": "Employment",
    "insurance_status": "Insurance",
    "is_hematologic": "Cancer Type",
    "comorbidity_score_capped": "Comorbidity Count (N)",
    "outcome_crn": "Outcome 1: Cost-Related Non-Adherence",
    "outcome_foregone": "Outcome 2: Foregone Medical Care",
    "outcome_subjective": "Outcome 3: Financial Worry",
    "age": "Age, years",
    "years_since_dx": "Years since diagnosis",
}
df_master = df_master.rename(columns=rename_map)

df_master["Household Income"] = df_master["Household Income"].replace({
    "Low": "<$50k",
    "Not Low": "≥50k",
})

df_master["Cancer Type"] = df_master["Cancer Type"].replace({
    0: "Solid",
    1: "Hematologic",
})

df_master["Sex"] = (
    df_master["Sex"]
    .astype(str).str.strip().str.lower()
    .replace({"female": "Female", "male": "Male", "unknown": "Unknown", "nan": "Unknown"})
)

OUTCOME_COLS = [
    "Outcome 1: Cost-Related Non-Adherence",
    "Outcome 2: Foregone Medical Care",
    "Outcome 3: Financial Worry",
]

for c in OUTCOME_COLS:
    df_master[c] = (
        pd.to_numeric(df_master[c], errors="coerce")
        .fillna(0)
        .astype(int)
        .astype(str)
    )

columns_base = [
    "Age, years",
    "Sex",
    "Nativity",
    "Marital Status",
    "Education",
    "Household Income",
    "Employment",
    "Insurance",
    "Cancer Type",
    "Years since diagnosis",
    "Comorbidity Count (N)",
    "Outcome 1: Cost-Related Non-Adherence",
    "Outcome 2: Foregone Medical Care",
    "Outcome 3: Financial Worry",
]

categorical_base = [c for c in columns_base if c not in ["Age, years", "Years since diagnosis"]]
nonnormal = ["Age, years", "Years since diagnosis"]

order_map = {
    "Sex": ["Female", "Male", "Unknown"],
    "Nativity": ["Foreign-born", "US-born", "Unknown"],
    "Marital Status": ["Not partnered", "Partnered", "Unknown"],
    "Education": [">HS", "≤HS", "Unknown"],
    "Household Income": ["<$50k", "≥50k", "Unknown"],
    "Employment": ["Not working", "Working", "Unknown"],
    "Insurance": ["Insured", "Uninsured", "Unknown"],
    "Cancer Type": ["Solid", "Hematologic", "Unknown"],
    "Comorbidity Count (N)": ["0", "1", "2", "3+", "Unknown"],
    "Outcome 1: Cost-Related Non-Adherence": ["1", "0"],
    "Outcome 2: Foregone Medical Care": ["1", "0"],
    "Outcome 3: Financial Worry": ["1", "0"],
}

limit_map = {
    "Outcome 1: Cost-Related Non-Adherence": 1,
    "Outcome 2: Foregone Medical Care": 1,
    "Outcome 3: Financial Worry": 1,
}

label_suffix = {c: f"{c}" for c in categorical_base}
label_suffix["Age, years"] = "Age, years"
label_suffix["Years since diagnosis"] = "Years since diagnosis"

# ==========================================================
#  TABLE 1A (Aggregate)
# ==========================================================

df_agg_pd = df_master.copy()
df_agg_pd["Cohort"] = df_agg_pd["Derived_Ethnicity"].apply(
    lambda x: "Asian American (Aggregate)" if x != "Non-Hispanic White" else "Non-Hispanic White"
)

table_1a = TableOne(
    df_agg_pd,
    columns=columns_base,
    categorical=categorical_base,
    nonnormal=nonnormal,
    groupby="Cohort",
    rename=label_suffix,
    order=order_map,
    limit=limit_map,
    pval=False,
    smd=True,
    overall=True,
    missing=False,
)

print("\n--- TABLE 1A (NO CENSORING) ---\n")
print(table_1a.tabulate(tablefmt="html"))

# ==========================================================
#  TABLE 1B (Subgroups)
# ==========================================================

df_sub_pd = df_master.copy()
df_sub_pd = df_sub_pd[df_sub_pd["Derived_Ethnicity"] != "Non-Hispanic White"]
df_sub_pd["Cohort"] = df_sub_pd["Derived_Ethnicity"]

table_1b = TableOne(
    df_sub_pd,
    columns=columns_base,
    categorical=categorical_base,
    nonnormal=nonnormal,
    groupby="Cohort",
    rename=label_suffix,
    order=order_map,
    limit=limit_map,
    pval=False,
    overall=True,
    missing=False,
)

print("\n--- TABLE 1B (NO CENSORING) ---\n")
print(table_1b.tabulate(tablefmt="html"))


# In[8]:


import polars as pl
import pandas as pd
import statsmodels.api as sm
import numpy as np
from plotnine import *
from great_tables import GT, md, html, style, loc
import warnings

warnings.filterwarnings('ignore')

from plotnine import options
options.figure_format = 'retina'
options.dpi = 900

ALL_ASIAN_GROUPS = ["Chinese", "Filipino", "Japanese", "Korean", "Vietnamese", "IndoPak"]

ANALYSIS_GROUPS = ["Chinese", "Filipino", "Japanese", "Korean", "Vietnamese", "IndoPak"]

OUTCOMES = [
    ("Cost-Related Non-Adherence", "outcome_crn"),
    ("Foregone Care", "outcome_foregone"),
    ("Financial Worry", "outcome_subjective")
]

ETHNICITY_LABELS = {
    "Chinese": "Chinese",
    "Filipino": "Filipino",
    "Japanese": "Japanese",
    "Korean": "Korean",
    "Vietnamese": "Vietnamese",
    "IndoPak": "Indian/Pakistani",
    "any_asian": "All Asian American",
    "is_multiracial": "Multiracial Identity (Adjustment)"
}



def prepare_analytic_cohort(df_polars):
    """Unified data preparation for all regression tables and plots."""
    valid_all_groups = [g for g in ALL_ASIAN_GROUPS if g in df_polars.columns]
    
    specific_groups = [g for g in ["Chinese", "Filipino", "Japanese", "Korean", "Vietnamese", "IndoPak"] if g in df_polars.columns]
    
    asian_mask = pl.max_horizontal(valid_all_groups) > 0
    
    df = (
        df_polars
        .with_columns(asian_mask.cast(pl.Int8).alias("any_asian"))
        .with_columns(pl.sum_horizontal(specific_groups).alias("asian_specific_count"))
        .with_columns(
            ((pl.col("asian_specific_count") > 1) | 
             ((pl.col("any_asian") == 1) & (pl.col("White") == 1))).cast(pl.Int8).alias("is_multiracial")
        )
        .filter((pl.col("White") == 1) | (pl.col("any_asian") == 1))
    )
    
    pdf = df.to_pandas()
    
    pdf["sex_male"] = (pdf["sex_normalized"] == "male").astype(int)
    pdf["unknown_sex"] = (pdf["sex_normalized"] == "unknown").astype(int)

    pdf["years_since_dx"] = pd.to_numeric(pdf["years_since_dx"], errors='coerce')
    pdf["is_hematologic"] = pdf["is_hematologic"].astype(int)
    pdf["age"] = pdf["age"].astype(float)
    
    pdf["low_income"] = (pdf["income_bracket"] == "Low").astype(int)
    pdf["unknown_income"] = (pdf["income_bracket"] == "Unknown").astype(int)

    pdf["uninsured"] = (pdf["insurance_status"] == "Uninsured").astype(int)
    
    pdf["low_education"] = (pdf["education_level"] == "≤HS").astype(int)
    pdf["unknown_education"] = (pdf["education_level"] == "Unknown").astype(int)

    pdf["not_working"] = (pdf["employment_status"] == "Not working").astype(int)
    pdf["unknown_employment"] = (pdf["employment_status"] == "Unknown").astype(int)

    pdf["foreign_born"] = (pdf["nativity"] == "Foreign-born").astype(int)

    pdf["not_partnered"] = (pdf["marital_status"] == "Not partnered").astype(int)
    pdf["unknown_marital"] = (pdf["marital_status"] == "Unknown").astype(int)

    if "comorbidity_score_capped" in pdf.columns:
        pdf["comorbidity_score_capped"] = pdf["comorbidity_score_capped"].fillna("0")
        dummies = pd.get_dummies(pdf["comorbidity_score_capped"], prefix="comorb", drop_first=True)
        pdf = pd.concat([pdf, dummies.astype(int)], axis=1)

    return pdf


def run_logistic_regression(data, outcome, predictors):
    """Unified regression model execution. Returns a DataFrame of all coefficients."""
    df_mod = data[[outcome] + predictors].dropna()
    if len(df_mod) < 50: 
        return None
    
    X = sm.add_constant(df_mod[predictors].astype(float))
    y = df_mod[outcome].astype(int)
    
    try:
        model = sm.Logit(y, X).fit(disp=False, method='bfgs', maxiter=2000)
        
        results = []
        for var_name in model.params.index:
            if var_name == 'const': continue
            results.append({
                'Variable': var_name,
                'OddsRatio': np.exp(model.params[var_name]),
                'LowerCI': np.exp(model.conf_int().loc[var_name, 0]),
                'UpperCI': np.exp(model.conf_int().loc[var_name, 1]),
                'PValue': model.pvalues[var_name]
            })
        return pd.DataFrame(results)
    except:
        return None


def get_targeted_stats(data, outcome, predictors, groups_to_extract):
    """Helper to extract specifically targeted groups for Standard Tables/Plots."""
    full_res = run_logistic_regression(data, outcome, predictors)
    if full_res is None or full_res.empty:
        return None
    
    res = full_res[full_res['Variable'].isin(groups_to_extract)].copy()
    res['Ethnicity'] = res['Variable'].apply(lambda x: ETHNICITY_LABELS.get(x, x))
    res = res.drop(columns=['Variable'])
    return res



def make_ultra_crisp_plot(all_data):
    if not all_data: return None
    
    combined_df = pd.concat(all_data, ignore_index=True)
    avg_upper_ci = combined_df.groupby('Ethnicity')['UpperCI'].mean().sort_values(ascending=False)
    combined_df['Ethnicity'] = pd.Categorical(combined_df['Ethnicity'], categories=avg_upper_ci.index, ordered=True)
    
    def get_color(row):
        if row['PValue'] < 0.05: return "red" if row['OddsRatio'] > 1 else "forestgreen"
        return "black"
    
    combined_df['Color'] = combined_df.apply(get_color, axis=1)
    p = (
        ggplot(combined_df, aes(y='Ethnicity', x='OddsRatio', color='Color'))
        + geom_point()
        + geom_errorbarh(aes(xmin='LowerCI', xmax='UpperCI', color='Color'), height=0.2)
        + geom_vline(xintercept=1, linetype="dotted")
        + scale_color_identity()
        + facet_wrap('~Outcome', ncol=3)
        + labs(x="Adjusted Odds Ratio", y="Race or Ethnicity")
        + theme_minimal()
        + theme(
            figure_size=(22, 8), dpi=900, legend_position="none",
            plot_title=element_text(size=22, weight="bold"),
            axis_title_x=element_text(size=22), axis_title_y=element_text(size=22),
            axis_text_x=element_text(size=18), axis_text_y=element_text(size=18),
            strip_text=element_text(size=28)  
        )
    )
    return p


def create_journal_table(all_data, title, ref_group):
    if not all_data: return None
    
    combined_df = pd.concat(all_data, ignore_index=True)
    outcomes = combined_df['Outcome'].unique()
    table_rows = []
    
    avg_upper_ci = combined_df.groupby('Ethnicity')['UpperCI'].mean().sort_values(ascending=False)
    ethnicities = avg_upper_ci.index.tolist()
    
    if "All Asian American" in ethnicities:
        ethnicities.remove("All Asian American")
        ethnicities.insert(0, "All Asian American")
        
    for ethnicity in ethnicities:
        row = {'Characteristic': ethnicity}
        for outcome in outcomes:
            subset = combined_df[(combined_df['Ethnicity'] == ethnicity) & (combined_df['Outcome'] == outcome)]
            if not subset.empty:
                r = subset.iloc[0]
                ci_text = f"{r['OddsRatio']:.2f} ({r['LowerCI']:.2f}, {r['UpperCI']:.2f})"
                raw_p = r['PValue']
                p_text = f"<b><0.001</b>" if raw_p < 0.001 else (f"<b>{raw_p:.3f}</b>" if raw_p < 0.05 else f"{raw_p:.3f}")
                row[f"{outcome}_aor"] = ci_text
                row[f"{outcome}_p"] = p_text
            else:
                row[f"{outcome}_aor"], row[f"{outcome}_p"] = "—", "—"
        table_rows.append(row)
        
    table_df = pd.DataFrame(table_rows)
    
    notes = [
        f"**Abbreviations:** aOR, adjusted Odds Ratio; CI, Confidence Interval.",
        f"**Reference Group:** {ref_group}.",
        f"Estimates were derived from multivariable logistic regression models.",
        f"Models are adjusted for age, sex, cancer type, time since diagnosis, income, insurance status, education, employment, nativity, marital status, multi-Asian ethnic identity, and comorbidity burden.",
        f"Bold values indicate statistical significance (two-sided P < 0.05)."
    ]

    return (
        GT(table_df)
        .tab_header(title=md(f"**{title}**"))
        .tab_spanner(label=md("**Cost-Related Non-Adherence**"), columns=[c for c in table_df.columns if 'Cost-Related' in c])
        .tab_spanner(label=md("**Foregone Care**"), columns=[c for c in table_df.columns if 'Foregone' in c])
        .tab_spanner(label=md("**Financial Worry**"), columns=[c for c in table_df.columns if 'Financial' in c])
        .cols_label(Characteristic=md("**Race/Ethnicity**"))
        .cols_label(**{c: md("aOR (95% CI)") if '_aor' in c else md("P-value") for c in table_df.columns if col != 'Characteristic'})
        .fmt_markdown(columns=[c for c in table_df.columns if '_p' in c])
        .tab_style(style=style.text(weight="bold"), locations=loc.body(columns="Characteristic"))
        .tab_style(style=style.fill(color="#f4f4f4"), locations=loc.body(rows=lambda d: d["Characteristic"] == "All Asian American"))
        .tab_options(table_font_size="13px", heading_title_font_size="14px", column_labels_font_size="12px", column_labels_font_weight="bold", data_row_padding="8px", table_width="100%")
        .tab_source_note(source_note=md(" ".join(notes)))
    )


def create_extended_table(pdf, predictors, group_vars, title, ref_group_name):
    ethnicity_rows = [(ETHNICITY_LABELS.get(g, g), "Race/Ethnicity", ref_group_name, g) for g in group_vars]
        
    covariate_rows = [
        ("Age (per year)", "Demographics", None, "age"),
        ("Male", "Demographics", "Female", "sex_male"),
        # Removed Sex Unknown as it is now imputed and entirely 0
        ("Foreign-Born", "Demographics", "US-Born", "foreign_born"),
        ("Not Partnered", "Demographics", "Partnered", "not_partnered"),
        # Removed Marital Status Unknown as it is now imputed and entirely 0
        ("Multiracial Identity", "Demographics", "Monoracial", "is_multiracial"),
        ("Low Income (<$50k)", "Socioeconomic Status", "High Income", "low_income"),
        ("Income Unknown", "Socioeconomic Status", None, "unknown_income"),
        ("Uninsured", "Socioeconomic Status", "Insured", "uninsured"),
        ("Education ≤ High School", "Socioeconomic Status", "Some College+", "low_education"),
        # Removed Education Unknown as it is now imputed and entirely 0
        ("Not Working", "Socioeconomic Status", "Working", "not_working"),
        # Removed Employment Unknown as it is now imputed and entirely 0
        ("Hematologic Cancer", "Clinical Characteristics", "Solid Tumor", "is_hematologic"),
        ("Years since diagnosis", "Clinical Characteristics", None, "years_since_dx"),
    ]

    comorb_rows = []
    for c in sorted([c for c in pdf.columns if c.startswith("comorb_")]):
        level = c.replace("comorb_", "")
        if level == "3": level = "3+"
        comorb_rows.append((f"Score {level}", "Comorbidity Burden", "Score 0", c))
        
    FULL_VAR_MAP = ethnicity_rows + covariate_rows + comorb_rows

    combined_results = {}
    for label, outcome_col in OUTCOMES:
        stats = run_logistic_regression(pdf, outcome_col, predictors)
        if stats is not None:
            combined_results[label] = stats.set_index('Variable')

    table_rows = []
    processed_headers = set()

    for display_name, category, ref_label, var_col in FULL_VAR_MAP:
        def make_row(name, is_ref=False):
            r_data = {"Characteristic": name, "Category": category, "Is_Ref": is_ref}
            for out_label, _ in OUTCOMES:
                if is_ref:
                    r_data[f"{out_label}_aor"], r_data[f"{out_label}_p"] = "1.00", "—"
                else:
                    res_df = combined_results.get(out_label)
                    if res_df is not None and var_col in res_df.index:
                        r = res_df.loc[var_col]
                        r_data[f"{out_label}_aor"] = f"{r['OddsRatio']:.2f} ({r['LowerCI']:.2f}, {r['UpperCI']:.2f})"
                        pv = r['PValue']
                        r_data[f"{out_label}_p"] = "<b><0.001</b>" if pv < 0.001 else (f"<b>{pv:.3f}</b>" if pv < 0.05 else f"<b>{pv:.3f}</b>" if pv < 0.05 else f"{pv:.3f}")
                    else:
                        r_data[f"{out_label}_aor"], r_data[f"{out_label}_p"] = "—", "—"
            return r_data

        if ref_label:
            unique_key = f"{category}_{ref_label}"
            if unique_key not in processed_headers:
                table_rows.append(make_row(f"{ref_label} (Reference)", is_ref=True))
                processed_headers.add(unique_key)
            table_rows.append(make_row(display_name))
        else:
            table_rows.append(make_row(display_name))

    df_tbl = pd.DataFrame(table_rows)
 
    notes = [
        f"**Abbreviations:** aOR, adjusted Odds Ratio; CI, Confidence Interval.",
        f"Estimates were derived from multivariable logistic regression models.",
        f"Bold values indicate statistical significance (two-sided P < 0.05)."
    ]
    
    return (
        GT(df_tbl, rowname_col="Characteristic", groupname_col="Category")
        .tab_header(title=md(f"**{title}**"))
        .tab_spanner(label=md("**Cost-Related Non-Adherence**"), columns=[c for c in df_tbl.columns if 'Cost-Related' in c])
        .tab_spanner(label=md("**Foregone Care**"), columns=[c for c in df_tbl.columns if 'Foregone' in c])
        .tab_spanner(label=md("**Financial Worry**"), columns=[c for c in df_tbl.columns if 'Financial' in c])
        .cols_label(Characteristic=md("**Characteristic**"))
        .cols_label(**{c: md("aOR (95% CI)") if '_aor' in c else md("P-value") for c in df_tbl.columns if '_aor' in c or '_p' in c})
        .fmt_markdown(columns=[c for c in df_tbl.columns if '_p' in c])
        .tab_style(style=style.fill(color="#f9f9f9"), locations=loc.body(rows=lambda d: d['Is_Ref']))
        .tab_options(row_group_font_weight="bold", row_group_background_color="#e0e0e0", table_font_size="12px", data_row_padding="5px", column_labels_font_weight="bold")
        .cols_hide(columns=["Is_Ref"])
        .tab_source_note(source_note=md(" ".join(notes)))
    )



def run_all_analyses():
    pdf = prepare_analytic_cohort(final_df)
    
    print("\n" + "="*95)
    print(f"ANALYSIS COHORT: N = {len(pdf):,}")
    print("="*95)
    
    # Removed imputed variables to prevent singular matrix error in the regressions
    unknown_cols = ["unknown_income"]
    
    base_full = ["age", "sex_male", "is_hematologic", "years_since_dx", "low_income", "uninsured", "low_education", "not_working", "foreign_born", "not_partnered", "is_multiracial"] + unknown_cols
    comorb_cols = [c for c in pdf.columns if c.startswith("comorb_")]
    vars_full = base_full + comorb_cols
    valid_analysis_groups = [g for g in ANALYSIS_GROUPS if g in pdf.columns]
    

    aggregate_data = []
    for label, outcome in OUTCOMES:
        res = get_targeted_stats(pdf, outcome, ["any_asian"] + vars_full, ["any_asian"])
        if res is not None:
            res['Outcome'] = label
            aggregate_data.append(res)
            
    mask_subgroups = (pdf['White'] == 1) | (pdf[valid_analysis_groups].sum(axis=1) > 0)
    pdf_sub = pdf[mask_subgroups].copy()
    
    subgroup_data = []
    for label, outcome in OUTCOMES:
        res = get_targeted_stats(pdf_sub, outcome, valid_analysis_groups + vars_full, valid_analysis_groups)
        if res is not None:
            res['Outcome'] = label
            subgroup_data.append(res)
            
    print("\n" + "="*95)
    print("FIGURE 1 & STANDARD TABLE 1. Asian Americans vs Non-Hispanic White")
    print("="*95 + "\n")
    if subgroup_data:
        print(make_ultra_crisp_plot(subgroup_data))
        t1_title = "Table 1: Associations (adjusted odds ratios) of race/ethnicity (vs NHW) with CRN, Foregone Care, and Financial Worry."
        display(create_journal_table(aggregate_data + subgroup_data, t1_title, "Non-Hispanic White"))

    print("\n" + "="*95)
    print("EXTENDED TABLE 1. Associations including all covariates")
    print("="*95 + "\n")
    t1_ext_title = "Extended Table 1: Full Model Associations of race/ethnicity (vs NHW) and covariates."
    display(create_extended_table(pdf_sub, valid_analysis_groups + vars_full, valid_analysis_groups, t1_ext_title, "Non-Hispanic White"))


    pdf_intra = pdf_sub[pdf_sub['any_asian'] == 1].copy()
    predictors_no_chinese = [g for g in valid_analysis_groups if g != "Chinese"]
    
    all_intra_data = []
    for label, outcome in OUTCOMES:
        res = get_targeted_stats(pdf_intra, outcome, predictors_no_chinese + vars_full, valid_analysis_groups)
        if res is not None:
            res['Outcome'] = label
            all_intra_data.append(res)
            
    print("\n" + "="*95)
    print("FIGURE 2 & STANDARD TABLE 2. Intra-Asian Comparison (Reference: Chinese)")
    print("="*95 + "\n")
    if all_intra_data:
        print(make_ultra_crisp_plot(all_intra_data))
        t2_title = "Table 2: Associations (adjusted odds ratios) of Asian subgroup (vs Chinese) with CRN, Foregone Care, and Financial Worry."
        display(create_journal_table(all_intra_data, t2_title, "Chinese"))

    print("\n" + "="*95)
    print("EXTENDED TABLE 2. Intra-Asian Comparison including all covariates")
    print("="*95 + "\n")
    t2_ext_title = "Extended Table 2: Full Model Associations of Asian subgroup (vs Chinese) and covariates."
    display(create_extended_table(pdf_intra, predictors_no_chinese + vars_full, predictors_no_chinese, t2_ext_title, "Chinese"))
    
    print("\n" + "="*95)
    print("ANALYSIS COMPLETE")
    print("="*95 + "\n")

if 'final_df' in locals():
    run_all_analyses()
else:
    print("Ensure 'final_df' (Polars DataFrame) is loaded.")


# In[9]:


import os
import polars as pl
import rpy2.robjects as ro
from IPython.display import Image, display

bucket_name = os.getenv("WORKSPACE_BUCKET").replace("gs://", "").strip("/")
path_base = f"gs://{bucket_name}/disagg/data"
phecode_path = f"gs://{bucket_name}/disagg/phecode/phecode_counts_X.tsv"

print("--- CALCULATING CONSORT ATTRITION NUMBERS ---")

lf_person = pl.scan_parquet(f"{path_base}/person_*.parquet")
n_step1_total = lf_person.select(pl.count()).collect().item()

lf_ehr = lf_person.filter(pl.col("has_ehr_data") == 1)
n_step2_ehr = lf_ehr.select(pl.count()).collect().item()
n_excl_no_ehr = n_step1_total - n_step2_ehr

lf_ft = pl.scan_parquet(f"{path_base}/ft_*.parquet")
ft_ids = lf_ft.select("person_id").unique()

lf_ehr_survey = lf_ehr.join(ft_ids, on="person_id", how="inner")
n_step3_survey = lf_ehr_survey.select(pl.count()).collect().item()
n_excl_no_survey = n_step2_ehr - n_step3_survey

lf_ft_dates = (
    lf_ft.lazy().group_by("person_id")
    .agg(pl.max("survey_datetime").alias("ft_index_date"))
    .with_columns(pl.col("ft_index_date").cast(pl.Date))
)

ehr_cancer_ids = (
    pl.scan_csv(phecode_path, separator="\t")
    .select(["person_id", "phecode", "first_event_date"])
    .with_columns(pl.col("first_event_date").str.to_date(strict=False).alias("dx_date"))
    .join(lf_ft_dates, on="person_id", how="inner")
    .filter((pl.col("ft_index_date") - pl.col("dx_date")).dt.total_days() >= 0)
    .filter(pl.col("phecode").str.starts_with("CA_") & ~pl.col("phecode").str.starts_with("CA_103.2"))
    .with_columns(pl.col("phecode").str.strip_prefix("CA_").cast(pl.Float64, strict=False).alias("p_num"))
    .filter(pl.col("p_num") < 135.0)
    .select("person_id")
    .unique()
)

lf_cancer_cohort = lf_ehr_survey.join(ehr_cancer_ids, on="person_id", how="inner")
n_step4_cancer = lf_cancer_cohort.select(pl.count()).collect().item()
n_excl_no_cancer = n_step3_survey - n_step4_cancer

VALID_ASIAN_ANSWERS = [
    'Asian Specific: Chinese', 
    'Asian Specific: Filipino', 
    'Asian Specific: Asian Specific Indian', 
    'Asian Specific: Pakistani',
    'Asian Specific: Japanese', 
    'Asian Specific: Korean', 
    'Asian Specific: Vietnamese'
]

lf_asian = (
    pl.scan_parquet(f"{path_base}/asian_*.parquet")
    .filter(pl.col("answer").is_in(VALID_ASIAN_ANSWERS))
    .select("person_id")
)
lf_white = pl.scan_parquet(f"{path_base}/white_*.parquet").select("person_id")

valid_race_ids = pl.concat([lf_asian, lf_white]).unique()

lf_final = lf_cancer_cohort.join(valid_race_ids, on="person_id", how="inner")
n_step5_final = lf_final.select(pl.count()).collect().item()

n_excl_race = n_step4_cancer - n_step5_final

ns = {
    "total": f"{n_step1_total:,}",
    "ehr": f"{n_step2_ehr:,}",
    "survey": f"{n_step3_survey:,}",
    "cancer": f"{n_step4_cancer:,}",
    "final": f"{n_step5_final:,}",
    "ex_ehr": f"{n_excl_no_ehr:,}",
    "ex_survey": f"{n_excl_no_survey:,}",
    "ex_cancer": f"{n_excl_no_cancer:,}",
    "ex_race": f"{n_excl_race:,}"
}

print("Counts Calculated:")
for k, v in ns.items():
    print(f"  {k}: {v}")

r_script = f"""
suppressPackageStartupMessages({{
  library(ggplot2)
  library(dplyr)
  library(tibble)
  library(ggtext)
  library(ggconsort)
  library(grid)
}})

# --- Build CONSORT structure ---
consort_data <- tibble(id = 1) %>%
  cohort_start("Start") %>%
  
  consort_box_add(
    "enrollment", 0, 100, 
    "**All of Us Research Program**<br>
    Total recruited participants<br>
    **(N = {ns['total']})**"
  ) %>%
  
  consort_box_add(
    "excl_1", 32, 90, 
    "**Excluded (n = {ns['ex_ehr']})**<br>
    No Electronic Health Record (EHR)<br>
    data linked to account"
  ) %>%
  
  consort_box_add(
    "ehr_step", 0, 80, 
    "**Participants with Linked EHR Data**<br>
    (n = {ns['ehr']})"
  ) %>%
  
  consort_box_add(
    "excl_2", 32, 70, 
    "**Excluded (n = {ns['ex_survey']})**<br>
    Did not complete required<br>
    program surveys:<br>
    &ndash; <i>Social Determinants of Health</i><br>
    &ndash; <i>Healthcare Access and Utilization</i>"
  ) %>%
  
  consort_box_add(
    "survey_step", 0, 60, 
    "**Completed Financial Toxicity**<br>
    **Screening Questions**<br>
    (n = {ns['survey']})"
  ) %>%
  
  consort_box_add(
    "excl_3", 32, 50, 
    "**Excluded (n = {ns['ex_cancer']})**<br>
    No diagnosis of malignant neoplasm<br>
    confirmed in EHR prior to survey"
  ) %>%
  
  consort_box_add(
    "cancer_step", 0, 40, 
    "**EHR-Confirmed Cancer Cohort**<br>
    (n = {ns['cancer']})"
  ) %>%
  
  consort_box_add(
    "excl_4", 32, 30, 
    "**Excluded (n = {ns['ex_race']})**<br>
    Race/Ethnicity other than<br>
    Asian (Chinese, Filipino, Indian,<br>
    Japanese, Korean, Vietnamese)<br>
    or Non-Hispanic White"
  ) %>%
  
  consort_box_add(
    "final_step", 0, 20, 
    "**Final Analytic Cohort**<br>
    (n = {ns['final']})"
  ) %>%
  
  consort_arrow_add(end = "ehr_step", end_side = "top", start_x = 0, start_y = 100) %>%
  consort_arrow_add(end = "survey_step", end_side = "top", start_x = 0, start_y = 80) %>%
  consort_arrow_add(end = "cancer_step", end_side = "top", start_x = 0, start_y = 60) %>%
  consort_arrow_add(end = "final_step", end_side = "top", start_x = 0, start_y = 40) %>%
  consort_arrow_add(end = "excl_1", end_side = "left", start_x = 0, start_y = 90) %>%
  consort_arrow_add(end = "excl_2", end_side = "left", start_x = 0, start_y = 70) %>%
  consort_arrow_add(end = "excl_3", end_side = "left", start_x = 0, start_y = 50) %>%
  consort_arrow_add(end = "excl_4", end_side = "left", start_x = 0, start_y = 30)

# --- Render plot ---
g <- consort_data %>%
  ggplot() + 
  geom_consort(label_width = NULL) +
  geom_richtext(
    data = . %>% filter(type == "box"),
    aes(x = x, y = y, label = label),
    fill = "white",
    label.color = "black",
    label.padding = unit(3, "mm"),
    label.r = unit(0, "pt"),
    family = "sans",
    size = 3.5,
    lineheight = 1.2
  ) +
  theme_void() +
  theme(
    plot.background = element_rect(fill = "white", color = NA),
    panel.background = element_rect(fill = "white", color = NA),
    plot.margin = margin(10, 50, 10, 10)
  ) +
  # EXPANDED LIMITS TO FIX CUTOFF
  coord_cartesian(xlim = c(-25, 100), ylim = c(5, 105))

ggsave("consort_final.png", plot = g, width = 8, height = 10, dpi = 300)
"""

print("Generating CONSORT Diagram...")
try:
    ro.r(r_script)
    if os.path.exists("consort_final.png"):
        display(Image(filename="consort_final.png"))
    else:
        print("Error: R script ran but file not found.")
except Exception as e:
    print(f"R Error: {e}")